# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con ollama_chat/gemma3:27b en poliGPT API

In [1]:
%pip install pydantic pandas langchain numpy pymupdf openai openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [21]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="gemma3:27b"
prefix = 'G20_2024_'
output_dir = "..\\Data\\Extracted Arguments Keywords (all text)\\"

## Input text processing

In [ ]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, keywords, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    # Keywords for filtering arguments
    positive_keywords = (keywords or {}).get('in_favor', [])
    negative_keywords = (keywords or {}).get('against', [])

    # Normalize & join for readability
    def to_str(xs):
        return ", ".join(sorted({s.strip().lower() for s in xs if isinstance(s, str) and s.strip()}))
    pos_kw_str = to_str(positive_keywords)
    neg_kw_str = to_str(negative_keywords)

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related ONLY to Sustainable Development Goal: {topic}\n"

            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. \n"
            "Your job is to identify and extract verbatim arguments about {topic} from long-form sustainability texts.\n\n"

            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Keywords for filtering:\n"
            "   - In favor: {pos_kw_str}\n"
            "   - Against: {neg_kw_str}\n"
            "4. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "5. If no qualifying arguments are found, return an empty array.\n\n"

            "Output Rules:\n"
            "   - Use only the exact text from the original\n"
            "   - No additional commentary or explanation\n"
            "   - Return only valid JSON\n"
            "   - No markdown formatting\n\n"

            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"

            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={
            "format_instructions": format_instructions,
            "pos_kw_str": pos_kw_str,
            "neg_kw_str": neg_kw_str,
        },
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    client = OpenAI(
    base_url = 'https://api.poligpt.upv.es',  
    api_key = 'sk-Icbf-5FyeV0QcLWBC9SNEA',
    timeout=180     
        )

    timeout = httpx.Timeout(60.0, connect=30.0) 

    chat_completion = client.chat.completions.create(
        messages = [
            {'role': 'system', 'content': 'You are an expert in logical reasoning, sustainability reporting, and argument analysis.'},
            {'role': 'user', 'content': final_prompt}
        ],
        model = model_name,
        temperature = 0,
        # timeout = timeout
    )

    raw_output = chat_completion.choices[0].message.content

    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, keywords, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, keywords, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic="", keywords=None):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, keywords, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", keywords=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic, keywords)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results

## SGD 1: Poverty

In [23]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"

keywords_g1 = {
    "in_favor": [
        "poverty reduction", "poverty alleviation", "social protection", "economic empowerment",
        "wealth creation", "opportunity", "prosperity", "development aid", "microfinance",
        "basic income", "empowerment", "upliftment", "sufficiency", "inclusion", "equity"
    ],
    "against": [
        "poverty", "pennilessness", "distress", "necessity", "hardship", "insolvency",
        "privation", "penury", "destitution", "hand-to-mouth existence", "beggary",
        "indigence", "pauperism", "necessitousness", "extreme poverty", "wealth inequality",
        "exploitation", "lack of opportunity", "exclusion", "vulnerability",
        "deprivation", "marginalization"
    ]
}


resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g1)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:
1. However, these important initiatives have been unilateral policy instruments that, largely, have not considered perspectives from the affected economies, potentially resulting in trade-offs (e.g., higher unemployment and poverty).

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 2: Hunger

In [24]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"
keywords_g2 = {
    "in_favor": [
        "food security", "food", "nutrition", "zero hunger", "nourishment",
        "food sovereignty", "food aid", "school feeding programs",
        "access to food", "healthy diets"
    ],
    "against": [
        "hunger", "undernutrition", "malnutrition", "starvation", "famine",
        "undernourishment", "food insecurity", "food waste", "crop failure",
        "land grabbing", "price volatility", "nutrient deficiency"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g2)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 3: Health

In [25]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"
keywords_g3 = {
    "in_favor": [
        "wellbeing", "welfare", "health", "benefit", "advantage", "comfort",
        "happiness", "prosperity", "universal health coverage", "healthcare access",
        "disease prevention", "mental health", "healthy lifestyles", "vaccination",
        "maternal health", "child health", "sanitation", "public health", "interest"
    ],
    "against": [
        "disease", "illness", "epidemic", "pandemic", "mortality", "morbidity",
        "health inequality", "stress", "poor sanitation", "addiction",
        "unhealthy habits", "mental illness", "anxiety"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g3)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 4: Education

In [26]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"
keywords_g4 = {
    "in_favor": [
        "quality", "inclusive", "equitable", "lifelong learning", "teaching",
        "schooling", "training", "development", "coaching", "instruction",
        "tutoring", "tuition", "skills development", "literacy", "numeracy",
        "universal access", "scholarships", 'data literacy'
    ],
    "against": [
        "lack of education", "illiteracy", "school dropout", "dropout",
        "educational inequality", "poor quality teaching", "indoctrination",
        "lack of access", "resource scarcity", "digital divide", "skills gap"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g4)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 5: Gender

In [27]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"
keywords_g5 = {
    "in_favor": [
        "gender equality", "women empowerment", "feminism", "women’s movement",
        "suffragette", "suffragist", "feminist", "emancipated", "equal rights",
        "equal opportunity", "women leadership", "girls education", "reproductive rights"
    ],
    "against": [
        "gender inequality", "sexism", "sexist", "discrimination", "gender violence",
        "misogyny", "patriarchy", "wage gap", "glass ceiling", "female genital mutilation",
        "child marriage", "lack of representation", "stereotypes", "glass ceiling"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g5)


merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:
1. Developed within the Development Working Group, the Action Plans aim to enhance policy coherence and coordination across all G20 work streams for the SDGs achievement.
2. the 2023 Action Plan, under the Indian presidency, called for action in three key areas: digital transformation, gender equality, and just transitions.

--- Processing Page 11 ---
Extracted Arguments:


## SGD 6: Water and sanitation

In [28]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"
keywords_g6 = {
    "in_favor": [
        "clean water", "sanitation", "hygiene", "cleanliness", "sewerage",
        "drinking water", "water access", "water management", "water efficiency",
        "wastewater treatment", "water quality"
    ],
    "against": [
        "water scarcity", "water pollution", "lack of sanitation", "open defecation",
        "waterborne diseases", "drought", "unsustainable water use", "contaminated water"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g6)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 7: Clean Energy

In [29]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"
keywords_g7 = {
    "in_favor": [
        "clean energy", "green energy", "renewable energy", "sustainable energy",
        "modern energy", "energy access", "energy efficiency", "solar power",
        "wind power", "geothermal energy", "hydropower", "energy transition", "energy matrix"
    ],
    "against": [
        "fossil fuels", "energy poverty", "energy inefficiency", "pollution",
        "carbon emissions", "unsustainable energy", "reliance on non-renewables"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g7)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:
1. The first JETP, launched at the COP-26, mobilizes $8.5 billion to 1) replace coal with renewable energy; 2) expand electric vehicles use; and 3) establish a green hydrogen industry in South Africa.

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 8: Decent Work, Economic Growth

In [30]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"
keywords_g8 = {
    "in_favor": [
        "decent work", "full employment", "fair wages", "workers rights",
        "job creation", "entrepreneurship", "financial inclusion", "financial",
        "business", "trade", "industrial", "commercial", "mercantile", "spillover"
    ],
    "against": [
        "unemployment", "underemployment", "precarious work", "exploitation",
        "child labor", "forced labor", "unsafe working conditions", "stagnation",
        "recession", "inequality", "informal economy", "low wages", "job insecurity",
        "informal jobs"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g8)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:
1. These are undesirable external economic, social, environmental, and security effects from a country’s actions which undermine other countries’ efforts to achieve the SDGs.
2. All countries, and especially the G20 given their combined economic weight, must better understand, monitor and manage their spillovers to allow all countries to meet the SDGs.

--- Processing Page 3 ---
Extracted Arguments:
1. The G20 is therefore the right forum for overcoming structural obstacles towards the SDGs such as negative spillover effects.

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:
1. trade-related spillovers are now measured through Consumption Based Accounting (CBA; Wood et al., 2020).
2. Ultimately, curbing negative spillovers requires multilateral solutions, especially between resource-rich and labor-rich countries with high income consumer countrie

## SGD 9: Infrastructure, industrilization, innovation

In [31]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"
keywords_g9 = {
    "in_favor": [
        "infrastructure", "industrialization", "innovation", "technological innovations",
        "research and development", "technology transfer", "connectivity", "internet access",
        "manufacturing", "scientific research", "digitalization", "modernization",
        "technological advances", "digital inclusion", "digital literacy", "technological investment" 
    ],
    "against": [
        "lack of infrastructure", "inadequate infrastructure", "industrial pollution",
        "unsustainable industry", "digital divide", "lack of innovation", "technological gap",
        "brain drain", "resource depletion", "unmaintained", "obsolescence", "decay", "cybersecurity threaths",
        "cybersecurity attacks"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g9)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 10: Inequality

In [32]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"
keywords_g10 = {
    "in_favor": [
        "equality", "equity", "inclusion", "equal opportunity", "fairness",
        "social justice", "progressive taxation", "non-discrimination"
    ],
    "against": [
        "inequality", "disparity", "discrimination", "exclusion", "apartheid",
        "linguistic imperialism", "favouritism", "bias", "partiality", "injustice",
        "imbalance", "nepotism", "marginalization", "wealth concentration",
        "poverty gap", "social stratification", "prejudice"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g10)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:
1. Negative spillovers may reinforce existing international injustices and divergences between countries of different income categories and regions.

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 11: Sustainable cities

In [33]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"
keywords_g11 = {
    "in_favor": [
        "sustainable cities", "sustainable communities", "smart cities", "urban planning",
        "affordable housing", "public transport", "green spaces", "community",
        "preservation", "society", "people", "public", "association", "population",
        "residents", "commonwealth", "general public", "spatial justice", "accessibility"
    ],
    "against": [
        "slums", "urban sprawl", "air pollution", "noise pollution", "traffic",
        "lack of housing", "urban poverty", "crime", "segregation", "gentrification",
        "unsafe", "insecure", "urban degradation", "housing crisis", "urban decay",
        "deteriorated urban areas", "disadvantaged communities"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g11)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---


APITimeoutError: Request timed out.

## SGD 12: Responsible Consumption, Responsible Production

In [ ]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"
keywords_g12 = {
    "in_favor": [
        "sustainable consumption", "sustainable production", "second use", "second hand",
        "circular economy", "recicle", "recycling", "reuse", "sustainable sourcing",
        "eco-design", "corporate social responsibility", "sustainable tourism",
        "manufacture", "manufacturing", "construction"
    ],
    "against": [
        "overconsumption", "waste", "using up", "expenditure", "exhaustion", "depletion",
        "dissipation", "pollution", "planned obsolescence", "fast fashion", "food waste",
        "unsustainable production", "resource inefficiency", "long-tail economy"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g12)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. incentivising decarbonisation in both production and in consumption
2. promoting sustainable food systems and reducing food waste.

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:
1. promoting a circular economy approach (60%) is the most common programme to reduce food waste.

--- Processing Page 12 ---
Extracted Arguments:
1. o Develop a comprehensive circular economy strategy that incentivises circular food supply chains and encourages the purchase of goods and services from circular businesses.

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- 

## SGD 13: Climate change

In [ ]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"
keywords_g13 = {
    "in_favor": [
        "climate action", "mitigation", "adaptation", "resilience", "carbon neutrality",
        "decarbonization", "energy transition", "emissions reduction",
        "Paris Agreement", "climate policy"
    ],
    "against": [
        "climate change", "global warming", "greenhouse gas emissions", "CO2 emissions",
        "fossil fuels", "deforestation", "climate inaction", "climate denial",
        "extreme weather events", "sea-level rise", "environmental degradation"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g13)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. at least 105 of the 169 targets that underlie the 17 SDGs are contingent upon the active engagement of LRGs.
2. incentivising decarbonisation in both production and in consumption

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:
1. Enhance public transportation options and affordability (e.g. temporary reduction in ticket prices for those most in need) to counter the financial burden of increasing fuel prices and incentivise the usage of low-carbon mobility options to help meet climate objectives.
2. Incentivise decarbonisation both in production and consumption.
3. Set decarbonisation goals, standards and regulations and lead by example to decarbonise government-owned and -operated infrastructure to achieve SDG 7 at the local level.

--- Processing Page 13 ---
Extracted Arguments:
1. The agr

## SGD 14: Life bellow water

In [ ]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"
keywords_g14 = {
    "in_favor": [
        "ocean conservation", "marine conservation", "sustainable fishing",
        "marine protected areas", "ocean biodiversity", "ocean ecosystems", "biology",
        "marine biology", "ecosystem restoration"
    ],
    "against": [
        "overfishing", "marine pollution", "plastic pollution", "microplastics",
        "ocean acidification", "coral bleaching", "habitat destruction", "illegal fishing",
        "destructive fishing practices", "biodiversity loss", "eutrophication"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g14)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:
1. The planet dimension (SDGs 6 and 12-15), which comprises the SDGs on water, sustainable consumption and production, climate action and life on land and underwater, received a slightly lower priority (20%)

--- Proce

## SGD 15: Life on land

In [ ]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"
keywords_g15 = {
    "in_favor": [
        "land ecosystem", "agriculture", "ecosystem restoration", "forest",
        "stop desertification", "reverse land degradation", "conservation",
        "sustainable agriculture", "afforestation", "reforestation",
        "wildlife protection", "wildlife"
    ],
    "against": [
        "deforestation", "desertification", "land degradation", "biodiversity loss",
        "habitat loss", "poaching", "illegal wildlife trade", "invasive species",
        "soil erosion", "unsustainable agriculture", "soil pollution"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g15)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:
1. The planet dimension (SDGs 6 and 12-15), which comprises the SDGs on water, sustainable consumption and production, climate action and life on land and underwater, received a slightly lower priority (20%)

--- Proce

## SGD 16: Peace, Justice, Strong Institutions

In [ ]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"
keywords_g16 = {
    "in_favor": [
        "peace", "justice", "access to justice", "strong institutions", "healthy institutions",
        "accountability", "anti-corruption", "transparency", "governance", "human rights",
        "conflict resolution", "truce", "ceasefire", "treaty", "armistice", "pacification",
        "fairness","integrity",
        "honesty", "decency", "impartiality", "justness", "rightfulness",
        "strong leadership", "good leadership", "institutionalization", "government effort", 
        "public investments", "science-based policy"

    ],

    "against": [
        "conflict", "violence", "war", "insecurity", "injustice", "corruption", "bribery",
        "weak institutions", "lack of accountability", "impunity", "human rights violations",
        "discrimination", "crime", "illicit financial flows", "organized crime", "terrorism",
         "weak leadership", "autoritarism", "dictator"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g16)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. d. Governance challenges in implementing the 2030 Agenda.

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. Political leadership at the local and regional levels is the most important success factor in SDG implementation for subnational governments
2. Political leadership is considered the most relevant success factor for the localisation of the SDGs.

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- P

## SGD 17: Partnerships, sustainable development

In [ ]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"
keywords_g17 = {
    "in_favor": [
        "global partnership", "cooperation", "association", "alliance", "sharing",
        "union", "connection", "participation", "copartnership", "technology transfer",
        "capacity building", "international cooperation", 'positive spillover', 
        "transboundary", "coordination"
    ],
    "against": [
        "lack of cooperation", "isolationism", "protectionism", "insufficient funding",
        "debt", "policy incoherence", "data gaps", "weak monitoring", "non-participation",
        "aid dependency", "technological gatekeeping", "negative spillover"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g17)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:
1. Our two institutions take great pride in this collaborative endeavour and look forward to supporting local,  regional and national governments further on their journey towards achieving the SDGs for a more sustainable and resilient future.

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:
1. Boost political leadership for the SDGs by actively engaging in national and international city networks that enable peer-to-peer learning on the 2030 Agenda and help adopt the SDGs as a policy-making and monitoring framework.

--- Processing Page 12 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. only half of surveyed LRGs and terri

## SGD 0: Overarching terms

In [ ]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"
keywords_g0 = {
    "in_favor": [
        "Sustainability", "Sustainable Development Goal", "SDG", "Agenda 2030", "global goals", 
        "development", "progress", "implementation", "monitoring", "accountability", "inclusive", "leave no one behind", 
        "Voluntary National Review", "VNR", "SDG transformations"
    ],
    "against": [
          "Unsustainability", "inaction", "regression", "lack of funding", "greenwashing", "exploitation", 
          "environmental degradation", "SDG needs", "regression", 
          "multidimensional vulnerability", "stagnation"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g0)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 4 ---
Extracted Arguments:
1. Only 15% of them are considered to be on track for achievement by the 2030 deadline.
2. Progress towards 48% of the SDG targets is currently insufficient, and 37% are either stagnating or regressing, including on key targets such as those related to poverty, hunger and climate action.
3. Cities and regions play a pivotal role in steering the SDGs back on track.
4. at least 105 of the 169 targets that underlie the 17 SDGs are contingent upon the active engagement of LRGs.

--- Processing Page 5 ---
Extracted Arguments:
1. Our two institutions take great pride in this collaborative endeavour and look forward to supporting local,  regional and national governments further on their journey towards achieving the SDGs for a more sustainable and resilient future.

--- Processing Page 10 ---
Extracted Arguments:
1. Findings show that local and regional governments (LRGs) are actively engaged in the implementation of the SDGs, notably through a